In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
SILVER_FORK_TABLE = f"{CATALOG}.{SCHEMA}.silver_fork_events"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.oss_ecosystem.silver_fork_events (
    event_id STRING,
    event_ts TIMESTAMP,
    actor_id BIGINT,
    repo_id BIGINT,
    forkee_repo_id BIGINT,
    forkee_repo_name STRING,
    source_file STRING,
    source_hour TIMESTAMP,
    ingested_at TIMESTAMP
)
USING DELTA
""")

In [0]:
fork_payload_schema = StructType([
    StructField(
        "forkee",
        StructType([
            StructField("id", LongType(), True),
            StructField("full_name", StringType(), True)
        ]),
        True
    )
])

In [0]:
fork_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "ForkEvent")
)

print(
    "ForkEvent rows in Bronze:",
    fork_bronze_df.count()
)

In [0]:
existing_fork_ids = (
    spark.table(SILVER_FORK_TABLE)
    .select("event_id")
)

new_fork_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "ForkEvent")
    .join(
        existing_fork_ids,
        on="event_id",
        how="left_anti"
    )
)

print(
    "New ForkEvents to transform:",
    new_fork_bronze_df.count()
)

In [0]:
silver_fork_df = (
    new_fork_bronze_df
    .withColumn(
        "payload_parsed",
        F.from_json(
            F.col("payload_json"),
            fork_payload_schema
        )
    )
    .select(
        "event_id",

        F.to_timestamp("created_at")
            .alias("event_ts"),

        F.get_json_object("actor_json", "$.id")
            .cast("long")
            .alias("actor_id"),

        F.get_json_object("repo_json", "$.id")
            .cast("long")
            .alias("repo_id"),

        F.col("payload_parsed.forkee.id")
            .alias("forkee_repo_id"),

        F.col("payload_parsed.forkee.full_name")
            .alias("forkee_repo_name"),

        "source_file",
        "source_hour",
        "ingested_at"
    )
)

In [0]:
print(
    "Silver Fork rows:",
    silver_fork_df.count()
)

silver_fork_df.select(
    F.sum(F.col("event_id").isNull().cast("int"))
        .alias("null_event_id"),

    F.sum(F.col("actor_id").isNull().cast("int"))
        .alias("null_actor_id"),

    F.sum(F.col("repo_id").isNull().cast("int"))
        .alias("null_repo_id"),

    F.sum(F.col("forkee_repo_id").isNull().cast("int"))
        .alias("null_forkee_repo_id"),

    F.sum(F.col("forkee_repo_name").isNull().cast("int"))
        .alias("null_forkee_repo_name")
).show()

In [0]:
duplicate_count = (
    silver_fork_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate ForkEvent IDs:",
    duplicate_count
)

In [0]:
if not spark.catalog.tableExists(SILVER_FORK_TABLE):
    (
        silver_fork_df.limit(0)
        .write
        .format("delta")
        .saveAsTable(SILVER_FORK_TABLE)
    )

silver_fork_delta = DeltaTable.forName(
    spark,
    SILVER_FORK_TABLE
)

(
    silver_fork_delta.alias("t")
    .merge(
        silver_fork_df.alias("s"),
        "t.event_id = s.event_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver Fork MERGE completed.")

In [0]:
print(
    "Silver Fork table rows:",
    spark.table(SILVER_FORK_TABLE).count()
)

print(
    "Duplicate ForkEvent IDs:",
    spark.table(SILVER_FORK_TABLE)
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)